# SkylineGeolocation — GSV Mask Retraining (DualHead U-Net)

Target: fix GSV mask quality (true-VP FB 0.65 -> goal >=0.9). Findings 2026-08-13:
threshold sweep + reliability weighting are dead ends; the U-Net follows cloud bases on
real GSV photos. This trains a DualHeadUnet (mask + per-column confidence) with
cloud-band augmentation and old-model finetune init.

Runtime: ~30 epochs, batch 8, T4 GPU ~30-45 min.

In [ ]:
# --- Cell 1: connect Drive + prep dirs ---
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/data /content/repo
print('Drive mounted')

In [ ]:
# --- Cell 2: clone repo (shallow) ---
import os
if not os.path.isdir('/content/repo/SkylineGeolocation'):
    !git clone --depth 1 https://github.com/pxrxp/SkylineGeolocation /content/repo/SkylineGeolocation
# NOTE: verify the repo URL; or upload the repo as a zip to Drive and unzip instead.
!ls /content/repo/SkylineGeolocation | head

In [ ]:
# --- Cell 3: copy data from Drive into /content/data ---
# Expected on Drive:  MyDrive/SkylineGeolocation_data/{geopose3k, clouds, synthetic_dataset, sky_segmentation_unet_model.pth}
import os
src = '/content/drive/MyDrive/SkylineGeolocation_data'
import shutil
for name in ['geopose3k', 'clouds', 'synthetic_dataset', 'sky_segmentation_unet_model.pth']:
    p = os.path.join(src, name)
    if os.path.exists(p):
        dst = f'/content/data/{name}'
        if not os.path.exists(dst):
            shutil.copytree(p, dst) if os.path.isdir(p) else shutil.copy(p, dst)
        print('copied', name)
    else:
        print('MISSING', name)

In [ ]:
# --- Cell 4: install deps ---
!pip install -q segmentation-models-pytorch albumentations timm fastdtw pyproj geopy rasterio pyarrow pandas
!pip install -q -e /content/repo/SkylineGeolocation 2>/dev/null || true
print('deps done')

In [ ]:
# --- Cell 5: train ---
import sys; sys.path.insert(0, '/content/repo/SkylineGeolocation')
import os
os.chdir('/content/repo/SkylineGeolocation')
!python scripts/retrain_dualhead.py \
    --geopose /content/data/geopose3k \
    --syn-img /content/data/synthetic_dataset/images \
    --syn-mask /content/data/synthetic_dataset/masks \
    --clouds /content/data/clouds \
    --init /content/data/sky_segmentation_unet_model.pth \
    --cloud-prob 0.7 \
    --epochs 30 --batch 8 --lr 1e-4 \
    --out /content/sky_segmentation_dualhead.pth

In [ ]:
# --- Cell 6: save best model to Drive ---
import shutil
best = '/content/sky_segmentation_dualhead.pth'
if os.path.exists(best):
    shutil.copy(best, '/content/drive/MyDrive/SkylineGeolocation_data/sky_segmentation_dualhead.pth')
    print('Saved to Drive:', os.path.getsize(best), 'bytes')
else:
    print('Training output not found')